In [2]:
import os
import re
import random
from glob import glob
import torch
import numpy as np
from datasets import load_dataset, Dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors
from transformers import (
    TrainerCallback,
    PreTrainedTokenizerFast,
    LlamaConfig,
    LlamaForCausalLM,
    AutoTokenizer,
    AutoModelForCausalLM,
)

from trl import SFTTrainer, SFTConfig

import warnings
warnings.filterwarnings("ignore", message=".*IProgress not found.*")


In [3]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(42)

### Предобработка данных

In [52]:
def preprocess_texts(folder_path, output_file, chunk_size):

    # Загружаем все строки из всех файлов
    all_lines = []
    for file_path in glob(os.path.join(folder_path, "*.txt")):
        with open(file_path, 'r', encoding='utf-8') as f:
            all_lines.extend(f.readlines())
    
    # Удаляем дубликаты строк 
    seen = set()
    unique_lines = []
    for line in all_lines:
        line_stripped = line.strip()
        if line_stripped and line_stripped not in seen:
            seen.add(line_stripped)
            unique_lines.append(line)
        elif not line_stripped:
            unique_lines.append(line)  
    
    # Оставляем только кириллицу, цифры, пробелы, базовую пунктуацию
    cleaned_lines = []
    for line in unique_lines:
        clean = re.sub(r'[^а-яА-ЯёЁ0-9\s\.\,\!\?\;\:\(\)\"\'\-\–\—]', '', line)
        # убираем лишние пробелы
        clean = re.sub(r'\s+', ' ', clean).strip()
        if clean:
            cleaned_lines.append(clean)
    
    # Склеиваем
    full_text = ' '.join(cleaned_lines)
    
    # Нормализуем повторяющуюся пунктуацию
    full_text = re.sub(r'\.{2,}', '.', full_text)
    full_text = re.sub(r'!{2,}', '!', full_text)
    full_text = re.sub(r'\?{2,}', '?', full_text)
    full_text = re.sub(r',{2,}', ',', full_text)
    
    # Разбиваем на чанки фиксированной длины
    chunks = []
    for i in range(0, len(full_text), chunk_size):
        chunk = full_text[i:i+chunk_size]
        if len(chunk) > 50:  # отбрасываем совсем короткие куски
            chunks.append(f"<bos> {chunk} <eos>")
    
    # Сохраняем чанки в файл
    with open(output_file, 'w', encoding='utf-8') as f:
        for chunk in chunks:
            f.write(chunk + '\n')
    
    return chunks

In [53]:
# Обработаем текст
texts = preprocess_texts("./RussianNovels/corpus", "./dataset/preprocessed_chunks.txt", chunk_size=1000)

In [54]:
# Посмотрим на пример
texts[0]

'<bos> КАПИТАНСКАЯ ДОЧКА Береги честь смолоду. Пословица. ГЛАВА СЕРЖАНТ ГВАРДИИ -- Был бы гвардии он завтра ж капитан. -- Того не надобно; пусть в армии послужит. -- Изрядно сказано! пускай его потужит. . . . . . . . . . . . . . . . Да кто его отец? Княжнин. Отец мой Андрей Петрович Гринев в молодости своей служил при графе Минихе и вышел в отставку премьер-майором в 17. году. С тех пор жил он в своей Симбирской деревне, где и женился на девице Авдотье Васильевне Ю., дочери бедного тамошнего дворянина. Нас было девять человек детей. Все мои братья и сестры умерли во младенчестве. Матушка была еще мною брюхата, как уже я был записан в Семеновский полк сержантом, по милости майора гвардии князя Б., близкого нашего родственника. Если бы паче всякого чаяния матушка родила дочь, то батюшка объявил бы куда следовало о смерти неявившегося сержанта, и дело тем бы и кончилось. Я считался в отпуску до окончания наук. В то время воспитывались мы не по-нонешнему. С пятилетнего возраста отдан я был

### Токенизатор

In [55]:
# Создаём BPE токенизатор
tokenizer = Tokenizer(models.BPE())

# Предтокенизация
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)

# Тренер с базовыми спецтокенами
trainer = trainers.BpeTrainer(vocab_size=3000, special_tokens=["<unk>", "<bos>", "<eos>", "<pad>"])

In [56]:
# Обучаем на созданном файле
tokenizer.train(["dataset/preprocessed_chunks.txt"], trainer=trainer)

In [57]:
# Добавляем постпроцессинг и декодер
tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)
tokenizer.decoder = decoders.ByteLevel()

# Сохраняем
tokenizer.save("models/bpe_tokenizer.json")

In [58]:
# Проверим размер словаря
print(f"Размер словаря: {tokenizer.get_vocab_size()}")

Размер словаря: 3000


### Токенизация

In [59]:
# Загружаем обученный BPE-токенизатор
tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="models/bpe_tokenizer.json",
    bos_token="<bos>",
    eos_token="<eos>",
    pad_token="<pad>",
    unk_token="<unk>"
)

In [60]:
# Токенизируем с длиной контекста 512 токенов
def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512     
    )

# Создаём Dataset из словаря
dataset = Dataset.from_dict({"text": texts})

# Применяем токенизацию ко всему датасету
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_dataset = tokenized_dataset.map(lambda x: {"labels": x["input_ids"]})

Map: 100%|██████████| 43008/43008 [00:15<00:00, 2811.39 examples/s]


In [61]:
# Преобразуем в формат PyTorch
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# Проверим результат
print(f"Размер датасета: {len(tokenized_dataset)}")
print(f"Пример input_ids: {tokenized_dataset[0]['input_ids'][:50]}...")

Размер датасета: 43008
Пример input_ids: tensor([   1,  259, 1576, 1706, 2441, 1991, 1576, 1279, 1894, 1909, 1576, 2663,
         285, 1863, 2854, 1909, 1576,  379,  140,  345,  105,  148, 1084,  381,
         968,  111,   11, 1367,  777, 1676,   11,  371,   61,   93, 1576, 1519,
        1576,  246, 2944,   61,   98,   61,   88, 1576, 1279, 1991,  371, 1519,
        1576,   61])...


### Обучение SFT

In [65]:
# Инициализируем модель Llama
config = LlamaConfig(
    hidden_size=1024,
    intermediate_size=1536,
    num_hidden_layers=16,
    num_attention_heads=16,
    num_key_value_heads=8,
    vocab_size=tokenizer.vocab_size,
    max_position_embeddings=512,
)
model = LlamaForCausalLM(config).cuda()
print(f"Количество параметров: {model.num_parameters():,}")

Количество параметров: 132,006,912


In [63]:
# Тестовые промпты
test_prompts = [
    "Все мысли, которые имеют огромные последствия",
    "Сила войска зависит от его духа",
    "Мысль о том, что он принес страдания",
    "Человек сознает себя свободным",
    "Что бы ни случилось, я всегда буду",
    "Любовь мешает смерти",
    "Нет, жизнь не кончена",
    "Всякая мысль, даже самая простая",
    "Война не любезность, а самое гадкое дело",
    "Чтобы жить честно"
]

In [ ]:
# Коллбэк для генерации на тестовых промптах после каждой эпохи
class GenerateCallback(TrainerCallback):
    def __init__(self, tokenizer, prompts, max_new_tokens=50):
        self.tokenizer = tokenizer
        self.prompts = prompts
        self.max_new_tokens = max_new_tokens

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        if model is None:
            return
        model.eval()
        print(f"\n===== Генерация после эпохи {state.epoch} =====")
        for prompt in self.prompts:
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=self.max_new_tokens,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id
                )
            print(f"Промпт: {prompt}\nГенерация: {tokenizer.decode(outputs[0], skip_special_tokens=True)}\n")
        model.train()

# Аргументы обучения
sft_config = SFTConfig(
    output_dir="./models",
    learning_rate=1e-4,
    per_device_train_batch_size=2,        
    gradient_accumulation_steps=1,      
    max_length=256,
    num_train_epochs=3,
    report_to="none",
    logging_steps=500,
    weight_decay=0.01,
    fp16=True,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    remove_unused_columns=False,           
    save_strategy="no",    
    packing=True               
)

# Trainer
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer,
    callbacks=[GenerateCallback(tokenizer, test_prompts)],
)

# Запуск обучения
print("Training device:", trainer.model.device)
trainer.train()

Padding-free training is enabled, but the attention implementation is not set to 'flash_attention_2'. Padding-free training flattens batches into a single sequence, and 'flash_attention_2' is the only known attention mechanism that reliably supports this. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation='flash_attention_2'` in the model configuration, or verify that your attention mechanism can handle flattened sequences.
You are using packing, but the attention implementation is not set to 'flash_attention_2' or 'kernels-community/vllm-flash-attn3'. Packing flattens batches into a single sequence, and Flash Attention is the only known attention mechanisms that reliably support this. Using other implementations may lead to cross-contamination between batches. To avoid this, either disable packing by setting `packing=False`, or set `attn_implementation='flash_attention_2'` or `attn_implementation='kernels-community/vllm-flash

Packing train dataset: 100%|██████████| 43008/43008 [00:01<00:00, 33247.90 examples/s]


Training device: cuda:0


Step,Training Loss
500,5.815300


In [ ]:
trainer.save_model("./pretrain_sft")

In [ ]:
def generate(model, tokenizer, prompt, max_new_tokens=100):
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    generated = input_ids
    with torch.no_grad():
        for _ in range(max_new_tokens):
            outputs = model(input_ids=generated)
            next_token_logits = outputs.logits[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
            generated = torch.cat((generated, next_token), dim=1)
            if next_token.item() == tokenizer.eos_token_id:
                break
    return tokenizer.decode(generated[0], skip_special_tokens=True)

for p in test_prompts:
    print(f"Prompt: {p}\nResponse:{generate(model, tokenizer, p)}\n")

## Post-train SFT

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B"          # базовая модель
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.chat_template = "{% for message in messages %}{% if message['role'] == 'user' %}{{ '<|im_start|>user\n' + message['content'] + '<|im_end|>\n' }}{% elif message['role'] == 'assistant' %}{{ '<|im_start|>assistant\n' + message['content'] + '<|im_end|>\n' }}{% endif %}{% endfor %}"

Some parameters are on the meta device because they were offloaded to the cpu and disk.


In [47]:
ds = load_dataset("d0rj/alpaca-cleaned-ru", split="train")

In [ ]:
def examples_to_messages(examples):
    data = {'messages': []}
    for example in examples:
        user_content = example['instruction']
        if example['input']:
            user_content += '\n' + example['input']
        data['messages'].append([
            {'role': 'user', 'content': user_content},
            {'role': 'assistant', 'content': example['output']}
        ])
    return Dataset.from_dict(data)

ds = examples_to_messages(ds)

In [ ]:
def formatting_func(example):
    messages = example.get('messages', [])
    if not messages:
        return [""]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    if not isinstance(text, str):
        text = str(text)
    return [text]

Map: 100%|██████████| 51760/51760 [00:09<00:00, 5224.30 examples/s]


In [ ]:
sft_config = SFTConfig(
    output_dir="./models",
    per_device_train_batch_size=2,     
    gradient_accumulation_steps=1,
    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=3,
    max_seq_length=256,
    logging_steps=50,
    save_steps=500,
    fp16=true,
    report_to="none",
    dataset_text_field="text",         
)

# Создание SFTTrainer 
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds,
    processing_class=tokenizer,
    formatting_func=formatting_func,
)

# Запуск обучения
# trainer.train()

# Сохранение модели
# trainer.save_model("./qwen_sft")

In [ ]:
questions = [
    "сколько планет в нашей солнечной системе?",
    "расскажи стих",
    "когда собирать крыжовник?",
    "Как быстро выучить новый язык?"
]

In [ ]:
def generate(model, tokenizer, prompt, max_new_tokens=100):
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    generated = input_ids
    with torch.no_grad():
        for _ in range(max_new_tokens):
            outputs = model(input_ids=generated)
            next_token_logits = outputs.logits[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
            generated = torch.cat((generated, next_token), dim=1)
            if next_token.item() == tokenizer.eos_token_id:
                break
    return tokenizer.decode(generated[0], skip_special_tokens=True)

for q in questions:
    print(f"Prompt: {q}\nResponce: {generate(model, tokenizer, q)}\n{'-'*50}")